# Build Your Own SRE Agent with Coral and Claude

Stand up a read-only SRE agent: Coral MCP for evidence, Claude via Pydantic AI for reasoning. All sources optional.

## Python setup

Run `./scripts/bootstrap.sh` from `SRE-agent/` if you haven't. The cell below loads `.env` and imports the widget helpers.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv('../.env')

from sre_agent.onboarding_ui import (
    anthropic_key_form,
    credentials_form,
    install_coral_form,
)

## 0. Install Coral

Coral is a local-first SQL interface for APIs ([docs](https://github.com/withcoral/coral)). Pick your OS:

```bash
# macOS
brew install withcoral/tap/coral
# Linux
curl -fsSL https://withcoral.com/install.sh | sh
```

In [ ]:
install_coral_form()

### Discover what Coral ships with

`coral source discover` lists every available source and verifies `coral` is on your PATH.

In [ ]:
!coral source discover

## 1. Enter credentials

Blank sections are skipped. All fields are required unless marked **(optional)**.

| Provider | Required env | Token source |
|---|---|---|
| Datadog | `DD_API_KEY`, `DD_APP_KEY` | [API keys](https://app.datadoghq.com/organization-settings/api-keys) |
| GitHub | `GITHUB_TOKEN` | [Fine-grained PAT](https://github.com/settings/personal-access-tokens) |
| Sentry | `SENTRY_TOKEN`, `SENTRY_ORG` | [Auth token](https://sentry.io/settings/account/api/auth-tokens/); org slug from URL |
| Slack | `SLACK_TOKEN` | Bot token (`xoxb-…`) |

In [ ]:
credentials_form()

## 2. Connect Coral to those sources

`coral source add` reads credentials from env. We only call it for providers you filled in above.

In [ ]:
if os.getenv("DD_API_KEY"):
    !coral source add datadog
if os.getenv("GITHUB_TOKEN"):
    !coral source add github
if os.getenv("SENTRY_TOKEN"):
    !coral source add sentry
if os.getenv("SLACK_TOKEN"):
    !coral source add slack

## 3. Check what's connected

Every source Coral has loaded — the agent's surface area.

In [ ]:
!coral source list

## 4. Run your first Coral query

Every source is queryable as SQL under `coral.tables`. This is the same call higher-level clients make under the hood.

In [ ]:
!coral sql "SELECT schema_name, COUNT(*) AS tables FROM coral.tables GROUP BY schema_name ORDER BY schema_name"

## 5. Use Coral from your AI coding agent

Coral is an MCP server — point any MCP-over-stdio client at `scripts/start_coral_mcp.sh` (it loads `.env` and picks the right Coral entrypoint).

**Claude Code** (from `SRE-agent/`):

```bash
claude mcp add coral --scope project -- ./scripts/start_coral_mcp.sh
```

Verify with `/mcp`, then ask: *"Using Coral, list Sentry issues from the last 24h with > 100 events."*

**Codex CLI** — add to `~/.codex/config.toml` (absolute path required):

```toml
[mcp_servers.coral]
command = "/ABSOLUTE/PATH/TO/SRE-agent/scripts/start_coral_mcp.sh"
```

## 6. Build a custom SRE agent with Pydantic AI *(optional)*

Skip this if §5 covers your use case. The cells below are the whole agent loop — Coral MCP as a toolset, Claude as the model, Pydantic AI for orchestration. The production version (retries, usage limits, Slack context) lives in `src/sre_agent/agent.py`.

Paste your [Anthropic API key](https://console.anthropic.com/):

In [ ]:
anthropic_key_form()

In [ ]:
from pydantic_ai import Agent, ModelSettings
from pydantic_ai.mcp import MCPServerStdio
from pydantic_ai.messages import FunctionToolCallEvent

from sre_agent.agent import SYSTEM_PROMPT
from sre_agent.coral_mcp import CoralMcpClient, load_coral_env


async def trace(ctx, events):
    """Print every Coral tool call as the agent runs."""
    async for event in events:
        if isinstance(event, FunctionToolCallEvent):
            print(f"🔧 {event.part.tool_name}({event.part.args})")


coral = CoralMcpClient()
agent = Agent(
    "anthropic:claude-sonnet-4-6",
    instructions=SYSTEM_PROMPT,
    toolsets=[MCPServerStdio(coral.coral_bin, args=coral.mcp_args, env=load_coral_env())],
    model_settings=ModelSettings(max_tokens=1800, temperature=0.0),
    event_stream_handler=trace,
)

In [ ]:
from IPython.display import Markdown

async with agent:
    result = await agent.run(
        "What SRE data sources can you see through Coral? "
        "For each, suggest one safe first investigation query."
    )

Markdown(result.output)